# 🎨 Paint-Code-RL: Zero-Cost GRPO Generative Art on Kaggle GPU

This notebook dynamically detects the Kaggle GPU runtime, configures headless WebGL via Node 20 LTS + Puppeteer Chrome + Xvfb, loads trained LoRA policies, and runs reinforcement learning training cycles.

### ⚡ Key Capabilities:
1. **Full Frame Canvas Capture**: Waits for `draw()` to finish execution so watercolor strokes never render blank.
2. **Procedural Image Fallbacks**: Automatically shims `loadImage` so texture hallucinations never stall rendering.
3. **Process Recycling**: Automatically flushes stale processes on port 3000 to ensure fresh code execution.
4. **Official Puppeteer Chrome**: Native Chrome binary installation bypassing Ubuntu's non-functional snap stub.
5. **Zero Conflicts**: Neutralizes Kaggle's incompatible `torchao` to prevent PEFT import errors.
6. **Tesla T4 Auto-Tuning**: Automatically selects `Qwen2.5-Coder-1.5B-Instruct` for 15.6GB VRAM.
7. **Instant Model Showcase**: Automatically downloads and evaluates `pernavjain/paint-code/pyTorch/default`.
8. **GRPO Cyclic Training**: Optimizes p5.js syntax and pixel-space visual richness with live dashboard tracking.
9. **One-Click Export**: Bundles all renders and checkpoints into a downloadable ZIP.

In [ ]:
# Cell 1: Environment & Hardware Verification & Memory Config
import torch, psutil, os

# Configure PyTorch memory allocator for maximum multi-GPU throughput and zero fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

print("=" * 60)
print("  PAINT-CODE-RL: ENVIRONMENT & HARDWARE VERIFICATION")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    device_count = torch.cuda.device_count()
    print(f"CUDA Devices Detected: {device_count}")
    total_vram = 0
    for i in range(device_count):
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        total_vram += vram
        print(f"  [GPU {i}] {torch.cuda.get_device_name(i)} ({vram:.1f} GB VRAM)")
    print(f"Total Combined VRAM: {total_vram:.1f} GB across {device_count} GPU(s)")
    if device_count >= 2:
        print("🚀 DUAL GPU SATURATION READY: Distributed training & generation active across both GPUs!")
else:
    print("  [WARN] Running on CPU. For fast GPU acceleration, enable GPU in Settings.")

ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1e9:.1f} GB Total / {ram.available / 1e9:.1f} GB Available")
print("=" * 60)

In [ ]:
# Cell 2: Install Node.js 20 LTS, Xvfb, and Chrome Runtime Libraries
# 1. Precompiled Node 20 LTS binary into /usr/local
!curl -fsSL https://nodejs.org/dist/v20.18.0/node-v20.18.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1

# 2. Remove any Ubuntu snap redirection stubs
!rm -f /usr/bin/chromium-browser /usr/bin/chromium

# 3. Linux X11/GL shared libraries required by headless Chrome
!apt-get update -qq && apt-get install -y -qq \
    libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 \
    libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 \
    xvfb > /dev/null 2>&1

import os
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"
!/usr/local/bin/node -v && /usr/local/bin/npm -v

In [ ]:
# Cell 3: Download & Extract Latest Repository Archive (Preserves node_modules & checkpoints)
import os, shutil

os.chdir("/kaggle/working")
!curl -fsSL "https://github.com/harshitthek/paint-code-rl/archive/refs/heads/main.zip" -o repo.zip
!unzip -q -o repo.zip
!mkdir -p paint-code-rl
!rsync -a --delete --exclude='renderer/node_modules' --exclude='artifacts' paint-code-rl-main/ paint-code-rl/
!rm -rf repo.zip paint-code-rl-main

os.chdir("/kaggle/working/paint-code-rl")
print("✅ Latest code loaded successfully at:", os.getcwd())


In [ ]:
# Cell 4: Install Python & Node.js Dependencies (with Puppeteer Chrome)
import os
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

# Neutralize Kaggle's incompatible pre-installed torchao 0.10.0 to prevent PEFT crash
!pip uninstall -y torchao > /dev/null 2>&1
!pip install -q trl==0.15.1 transformers==4.49.0 peft datasets accelerate pydantic safetensors Pillow pyyaml psutil requests kagglehub huggingface_hub

os.chdir("/kaggle/working/paint-code-rl/renderer")
!/usr/local/bin/npm install --no-audit --no-fund
!npx puppeteer browsers install chrome --install-deps
os.chdir("/kaggle/working/paint-code-rl")
print("[OK] Dependencies installed successfully!")


In [ ]:
# Cell 5: Launch Headless WebGL Rendering Daemon with Xvfb
import subprocess, time, requests, os, secrets
os.environ["PATH"] = f"/usr/local/bin:{os.environ.get('PATH', '')}"

# Cleanly stop tracked or existing renderer instance on port 3000
if 'proc' in globals() and proc and proc.poll() is None:
    try:
        proc.terminate()
        proc.wait(timeout=2)
    except Exception:
        proc.kill()
if 'RENDERER_SHUTDOWN_TOKEN' in os.environ:
    try:
        requests.post('http://127.0.0.1:3000/shutdown', headers={'X-Renderer-Token': os.environ['RENDERER_SHUTDOWN_TOKEN']}, timeout=1, allow_redirects=False)
        time.sleep(1)
    except Exception:
        pass
!rm -f /usr/bin/chromium-browser /usr/bin/chromium
os.environ["RENDERER_SHUTDOWN_TOKEN"] = os.environ.get("RENDERER_SHUTDOWN_TOKEN") or secrets.token_hex(16)
os.environ["CONCURRENT_WORKERS"] = str(max(4, (os.cpu_count() or 2) * 2))
time.sleep(1)

# Start renderer daemon under xvfb virtual frame buffer with max worker concurrency
env = os.environ.copy()
proc = subprocess.Popen(
    ["xvfb-run", "-s", "-screen 0 1920x1080x24 -ac +extension GLX +render -noreset", "/usr/local/bin/node", "renderer/server.js"],
    env=env
)
time.sleep(4)

# Verify renderer health
for attempt in range(5):
    try:
        health = requests.get("http://127.0.0.1:3000/health", timeout=3).json()
        print("[OK] Renderer daemon is live:", health)
        break
    except Exception:
        time.sleep(2)
else:
    print("[WARN] Renderer daemon did not respond immediately, continuing...")

In [ ]:
# Cell 6: Generate Artwork in MAX Power & Efficiency Mode (--max)
# Saturated Dual-GPU tensor sharding & concurrent WebGL workers
# Automatically loads trained local LoRA checkpoint or pulls from Hugging Face:
import os, glob
from IPython.display import display, Image

!python scripts/generate_and_render.py --output-dir artifacts/renders --temperature 0.4 --max-new-tokens 550 --max

renders = sorted(glob.glob("artifacts/renders/render_*.png"))
print(f"Total artworks generated: {len(renders)}")
for r in renders:
    print(f"File: {r}")
    display(Image(filename=r, width=400))

In [ ]:
# Cell 7: Run GRPO Cyclic Training on GPU with Auto Hardware Saturation
import os, torch
os.environ["ENV"] = "kaggle"
os.environ["PYTHONUNBUFFERED"] = "1"

# Automatically utilize Dual GPUs via torchrun DDP when available, with max hardware saturation
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
if n_gpus > 1:
    print(f"🚀 Detected {n_gpus} GPUs! Launching torchrun for distributed Dual-GPU training at --max...")
    !torchrun --nproc_per_node={n_gpus} --master_port=29500 scripts/train_grpo.py --mode train --steps-per-cycle 25 --max-steps 25 --unattended --max --dashboard
else:
    !python scripts/train_grpo.py --mode train --steps-per-cycle 25 --max-steps 25 --unattended --max --dashboard

In [ ]:
# Cell 8: View Live Dashboard Inline (Clean, Zero-Warning Display)
import os, warnings
from IPython.display import display, HTML

if os.path.exists("artifacts/dashboard.html"):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        with open("artifacts/dashboard.html", "r", encoding="utf-8") as f:
            html_code = f.read()
        display(HTML(f'<iframe srcdoc="{html_code.replace(chr(34), "&quot;")}" width="100%" height="600px" frameborder="0"></iframe>'))


In [ ]:
# Cell 9: Package All Outputs into Downloadable ZIP
# After running this, download 'paint_rl_artifacts.zip' from Kaggle's right-hand Output panel!
!python scripts/package_artifacts.py --output-dir /kaggle/working
print("\n[DONE] Check the right-hand panel under 'Output' to download your ZIP file!")

In [ ]:
# Cell 10: Upload Trained Model to Hugging Face Hub & Kaggle Models
import os

# Safely extract HF_TOKEN from Kaggle Secrets if configured
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("🔑 Retrieved HF_TOKEN from Kaggle Secrets successfully.")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")
    if not hf_token:
        print("ℹ️ HF_TOKEN not found in Kaggle Secrets. Set HF_TOKEN in Add-ons -> Secrets or provide below.")

# 1. Upload to Hugging Face Hub (uses HF_TOKEN from environment if --token is omitted):
token_arg = f"--token {hf_token}" if hf_token else ""
!python scripts/upload_model.py --destination hf --repo-id HarshittheK/paint-code-rl-lora {token_arg}

# 2. Upload to Kaggle Models (optional):
# !python scripts/upload_model.py --destination kaggle --handle harshitxdev/paint-code/pyTorch/v2 --version-notes "Trained via GRPO visual RL" 

In [ ]:
# Cell 11: Test Published Model Directly from Hugging Face Hub
# Loads and renders artworks using cloud-hosted LoRA adapter across Dual GPUs:
!python scripts/generate_and_render.py --hf HarshittheK/paint-code-rl-lora --output-dir artifacts/renders_hf --max